In [ ]:
import math
import os
import gurobipy as gp
import contextlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import random
import time
import itertools

from collections import deque
from typing import Optional, Dict, List
from openpyxl import load_workbook

class E2EVRP:
    def __init__(self, folder, name):
        # Read path
        instance_path = f"./e-2e-vrp instances/{folder}/{name}.txt"
        if not os.path.exists(instance_path):
            raise FileNotFoundError(f"{instance_path} was not found.")

        self.instance_name = name
        section = None

        # Read parameters

        # Nodes
        self.N = []
        self.Nd = None 
        self.NS = []
        self.NC = []
        self.NR = []
        self.coord = {}

        # Satellite parameters
        self.K = {}

        # Customer parameters
        self.q = {}
        self.S = {}
        self.e = {}
        self.l = {}
        self.p = {}

        # Vehicles
        self.V = []
        self.V1 = []
        self.V2 = []
        
        # Vehicle parameters
        self.m = {}
        self.Q = {}
        self.h = {}
        self.c = {}
        self.ms = {}
        self.L = {}
        self.r = {}

        # Constant parameters
        self.eta = None

        with open(instance_path, "r", encoding="utf-8") as f:
            for raw in f:
                line = raw.strip()

                if not line or line.startswith("!--"):
                    continue

                if line.startswith("!Stores"):
                    section = "stores"
                    continue
                if line.startswith("!Trucks"):
                    section = "trucks"
                    continue
                if line.startswith("!CityFreighters"):
                    section = "cityf"
                    continue
                if line.startswith("!Customers"):
                    section = "cust"
                    continue
                if line.startswith("!Recharge stations"):
                    section = "recharge"
                    continue

                if line.startswith("g "):
                    parts = line.strip().split()
                    self.eta = float(parts[-1])
                    continue

                if section == "stores":
                    items = line.split()
                    for idx, item in enumerate(items):
                        if idx == 0: # Deposit
                            x, y = item.split(",")
                            self.Nd = "D0"
                            self.coord[self.Nd] = (float(x), float(y))
                            self.N.append(self.Nd)
                        else: # Satellites
                            x, y, maxCap = item.split(",")
                            s_id = f"S{idx-1}"
                            self.coord[s_id] = (float(x), float(y))
                            self.K[s_id] = maxCap
                            self.N.append(s_id)
                            self.NS.append(s_id)
                elif section == "trucks":
                    items = line.split()
                    for idx, item in enumerate(items):
                        tn, tq, tc, th = item.split(",")
                        ttype = f"T{idx}"
                        self.m[ttype] = float(tn)
                        self.Q[ttype] = float(tq)
                        self.h[ttype] = float(th)
                        self.c[ttype] = float(tc)
                        self.V.append(ttype)
                        self.V1.append(ttype)
                elif section == "cityf":
                    items = line.split()
                    for idx, item in enumerate(items):
                        fns, fn, fq, fc, fh, fb, fr = item.split(",")
                        ftype = f"F{idx}"
                        self.m[ftype] = float(fn)
                        self.ms[ftype] = float(fns)
                        self.Q[ftype] = float(fq)
                        self.h[ftype] = float(fh)
                        self.c[ftype] = float(fc)
                        self.L[ftype] = float(fb)
                        self.r[ftype] = float(fr)
                        self.V.append(ftype)
                        self.V2.append(ftype)
                elif section == "cust":
                    items = line.split()
                    for idx, item in enumerate(items):
                        x, y, d, rt, dd, st, pc = item.split(",")
                        c_id = f"C{idx}"
                        self.coord[c_id] = (float(x), float(y))
                        self.q[c_id] = float(d)
                        self.e[c_id] = float(rt)
                        self.l[c_id] = float(dd)
                        self.S[c_id] = float(st)
                        self.p[c_id] = float(pc)
                        self.N.append(c_id)
                        self.NC.append(c_id)
                elif section == "recharge":
                    items = line.split()
                    for idx, item in enumerate(items):
                        x, y = item.split(",")
                        r_id = f"R{idx}"
                        self.coord[r_id] = (float(x), float(y))
                        self.N.append(r_id)
                        self.NR.append(r_id)
                else:
                    continue

        # Arc parameters
        self.d = {}
        self.dr = {}
        self.t = {}

        for i in self.N:
            xi, yi = self.coord[i]
            for j in self.N:
                xj, yj = self.coord[j]
                dij = round(math.sqrt((xi-xj)**2 + (yi-yj)**2))
                self.d[i,j] = dij
                self.t[i,j] = dij
        
        for i in self.NS+self.NC:
            xi, yi = self.coord[i]
            for j in self.NS+self.NC:
                self.dr[i,j,'0'] = self.d[i,j]
                for r in self.NR:
                    self.dr[i,j,r] = self.d[i,r] + self.d[r,j]

        # Matheuristic
        self.pq = 0.25
        self.max = 50
        self.op = [(r, i) for r in ['Rnd','Dist-r','W-Dist','TW-r','W-Lat']
                        for i in ['Rnd','Dist','s-TW','e-TW']]
        self.alpha = 0.005
        self.H = {o: 0 for o in self.op}
        self.R = 0
        self.n = 0

        # Routes
        self.routes = {}
        self.Rd = {}
        self.A = {}
        self.b = {}
        self.C = {}

    def milp_model(self, env_params, time_limit, output=0):
        M = 10**(math.ceil(math.log10(max(self.l.values())))+2)
        Mq = 10**(math.ceil(math.log10(max(self.Q.values())))+2)
        Ml = 10**(math.ceil(math.log10(max(self.L.values())))+2)

        N1 = [self.Nd]+self.NS
        N2 = self.NS+self.NC
        NR_ = ['0']+self.NR

        V1, V2, vt = [], [], {}
        VT1, VT2 = self.V1, self.V2
        vehT = {}

        n1 = 0
        for v in VT1:
            for _ in range(int(self.m[v])):
                vname = f"vT{n1}"
                V1.append(vname)
                vt[vname] = v
                n1 += 1

        n2 = 0
        for v in VT2:
            vehT[v] = []
            for _ in range(int(self.m[v])):
                vname = f"vF{n2}"
                V2.append(vname)
                vt[vname] = v
                vehT[v].append(vname)
                n2 += 1

        with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            env = gp.Env(params=env_params)

        # Model
        model = gp.Model(self.instance_name, env=env)
        model.Params.OutputFlag = output
        model.Params.TimeLimit = time_limit

        # Decision variables
        w = {(i,s): model.addVar(vtype = gp.GRB.BINARY, name = f"w({i},{s})") for i in self.NC for s in self.NS}
        a = {s: model.addVar(vtype = gp.GRB.BINARY, name = f"a({s})") for s in self.NS}
        x1 = {(i,j,v): model.addVar(vtype = gp.GRB.BINARY, name = f"x^D({i},{j},{v})") for i in N1 for j in N1 for v in V1 if i!=j}
        y1 = {v: model.addVar(vtype = gp.GRB.BINARY, name = f"y^D({v})") for v in V1}
        U1 = {(i,v): model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"U^({i},{v})") for i in N1 for v in V1}
        x2 = {(i,j,r,v,s): model.addVar(vtype = gp.GRB.BINARY, name = f"x^{s}({i},{j},{r},{v})") for s in self.NS for i in [s]+self.NC for j in [s]+self.NC if i!=j for r in NR_ for v in V2}
        y2 = {(v,s): model.addVar(vtype = gp.GRB.BINARY, name = f"y^{s}({v})") for s in self.NS for v in V2}
        U2 = {(i,v): model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"U^2({i},{v})") for i in N2 for v in V2}
        E = {(i,v): model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"E({i},{v})") for i in N2 for v in V2}
        T = {i: model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"T({i})") for i in N2}
        o = {i: model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"o({i})") for i in self.NC}

        # Objective function
        model.setObjective(gp.quicksum(self.d[i,j]*self.c[vt[v]]*x1[i,j,v] for i in N1 for j in N1 for v in V1 if i!=j) + gp.quicksum(self.h[vt[v]]*y1[v] for v in V1) + 
                           gp.quicksum(self.dr[i,j,r]*self.c[vt[v]]*x2[i,j,r,v,s] for s in self.NS for i in [s]+self.NC for j in [s]+self.NC if i!=j for r in NR_ for v in V2) +
                           gp.quicksum(self.h[vt[v]]*y2[v,s] for v in V2 for s in self.NS) + gp.quicksum(self.p[i]*o[i] for i in self.NC), gp.GRB.MINIMIZE)

        # Constraints

        # Assignament
        # Assign satellite with customer
        for i in self.NC:
            model.addConstr(gp.quicksum(w[i,s] for s in self.NS) == 1)

        # Visit satellite if a customer is assigned
        for s in self.NS:
            model.addConstr(gp.quicksum(self.q[i]*w[i,s] for i in self.NC) <= self.K[s]*a[s])

        # First echelon
        # Arrives and departures at satellites
        for s in self.NS:
            model.addConstr(gp.quicksum(x1[i,s,v] for i in N1 if i!=s for v in V1) == a[s])
            model.addConstr(gp.quicksum(x1[s,i,v] for i in N1 if i!=s for v in V1) == a[s])

        # Arrives and departures at deposit per route
        for v in V1:
            model.addConstr(gp.quicksum(x1[self.Nd,s,v] for s in self.NS) == y1[v])
            model.addConstr(gp.quicksum(x1[s,self.Nd,v] for s in self.NS) == y1[v])

        # Flow conservation at satellites
        for s in self.NS:
            for v in V1:
                model.addConstr(gp.quicksum(x1[i,s,v] for i in N1 if i!=s) - gp.quicksum(x1[s,j,v] for j in N1 if j!=s) == 0)

        # Transported demand
        for v in V1:
            model.addConstr(U1[self.Nd,v] == 0)

            for i in N1:
                model.addConstr(U1[i,v] <= self.Q[vt[v]]*y1[v])

                for j in self.NS:
                    if i!=j:
                        model.addConstr(U1[j,v] >= U1[i,v] + gp.quicksum(self.q[c]*w[c,j] for c in self.NC) - Mq*(1-x1[i,j,v]))

        # Second-echelon
        # A route cannot be assigned to two or more satellites
        for v in V2:
            model.addConstr(gp.quicksum(y2[v,s] for s in self.NS) <= 1)

        # Limited number of routes per satellite
        for s in self.NS:
            for v_ in VT2:
                model.addConstr(gp.quicksum(y2[v,s] for v in vehT[v_]) <= self.m[v_])

        # Arrives and departures at customers
        for i in self.NC:
            for s in self.NS:
                model.addConstr(gp.quicksum(x2[j,i,r,v,s] for j in [s]+self.NC if i!=j for r in NR_ for v in V2) == w[i,s])
                model.addConstr(gp.quicksum(x2[i,j,r,v,s] for j in [s]+self.NC if i!=j for r in NR_ for v in V2) == w[i,s])

        # Arrives and departures at satellites per route
        for v in V2:
            for s in self.NS:
                model.addConstr(gp.quicksum(x2[s,i,r,v,s] for i in self.NC for r in NR_) == y2[v,s])
                model.addConstr(gp.quicksum(x2[i,s,r,v,s] for i in self.NC for r in NR_) == y2[v,s])

        # Flow conservation at customers
        for i in self.NC:
            for s in self.NS:
                for v in V2:
                    model.addConstr(gp.quicksum(x2[j,i,r,v,s] for j in [s]+self.NC if i!=j for r in NR_) - gp.quicksum(x2[i,k,r,v,s] for k in [s]+self.NC if i!=k for r in NR_) == 0)

        # Transported demand
        for v in V2:
            for s in self.NS:
                model.addConstr(U2[s,v] == 0)

            for i in self.NC:
                model.addConstr(U2[i,v] <= self.Q[vt[v]]*gp.quicksum(y2[v,s] for s in self.NS))

            for s in self.NS:
                for i in [s]+self.NC:
                    for j in self.NC:
                        if i!=j:
                            for r in NR_:
                                model.addConstr(U2[j,v] >= U2[i,v] + self.q[j] - Mq*(1-x2[i,j,r,v,s]))

        # Energy level
        for v in V2:
            for s in self.NS:
                model.addConstr(E[s,v] == self.L[vt[v]]*y2[v,s])

            for i in self.NC:
                model.addConstr(E[i,v] <= self.L[vt[v]]*gp.quicksum(y2[v,s] for s in self.NS))

            for s in self.NS:
                for i in [s]+self.NC:
                    for j in self.NC:
                        if i!=j:
                            model.addConstr(E[j,v] <= E[i,v] - self.r[vt[v]]*self.d[i,j] + Ml*(1-x2[i,j,'0',v,s]))

                            for r in self.NR:
                                model.addConstr(E[i,v] >= self.r[vt[v]]*self.d[i,r]*x2[i,j,r,v,s])
                                model.addConstr(E[j,v] <= self.L[vt[v]] - self.r[vt[v]]*self.d[r,j] + Ml*(1-x2[i,j,r,v,s]))


            for i in self.NC:
                for s in self.NS:
                    model.addConstr(E[i,v] >= self.r[vt[v]]*self.d[i,s]*x2[i,s,'0',v,s])

                    for r in self.NR:
                        model.addConstr(self.L[vt[v]] >= self.r[vt[v]]*self.d[r,s]*x2[i,s,r,v,s])
                        model.addConstr(E[i,v] >= self.r[vt[v]]*self.d[i,r]*x2[i,s,r,v,s])

        # Arrive hours
        for i in self.NC:
            for v in V2:
                for s in self.NS:
                    model.addConstr(T[i] >= T[s] + self.t[s,i] - M*(1-x2[s,i,'0',v,s]))

                    for r in self.NR:
                        model.addConstr(T[i] >= T[s] + self.t[s,r] + self.eta*self.r[vt[v]]*self.d[s,r] + self.t[r,i] - M*(1-x2[s,i,r,v,s]))

                    for j in self.NC:
                        if i!=j:
                            model.addConstr(T[j] >= T[i] + self.S[i] + self.t[i,j] - M*(1-x2[i,j,'0',v,s]))

                            for r in self.NR:
                                model.addConstr(T[j] >= T[i] + self.S[i] + self.t[i,r] + self.eta*(self.L[vt[v]]-(E[i,v]-self.r[vt[v]]*self.d[i,r])) + self.t[r,j] - M*(1-x2[i,j,r,v,s]))

            model.addConstr(T[i] >= self.e[i])
            model.addConstr(T[i] <= self.l[i] + o[i])
                             
        model.update()
        model.optimize()

        if model.Status == gp.GRB.INFEASIBLE:
            objf = float("inf")
            gap = float("inf")
            exe_time = model.Runtime
        else:
            # Save 1st level routes
            # print("First level:")
            # for v in V1:
            #     if y1[v].X > 0.9:
            #         route = [self.Nd]
            #         s = self.Nd
            #         while True:
            #             next_node = None

            #             for i in [self.Nd] + self.NS:
            #                 if i != s:
            #                     if x1[s,i,v].X > 0.9:
            #                         next_node = i
            #                         break

            #             route.append(next_node)
            #             s = next_node
                        
            #             if s == self.Nd:
            #                 break
            #         print(v, route)

            # Save 2nd level routes
            # print("Second level:")
            # for s in self.NS:
            #     if a[s].X > 0.9:
            #         print(s)

            #         for v in V2:
            #             if y2[v,s].X > 0.9:
            #                 route = [s]
            #                 c = s
            #                 while True:
            #                     next_node = None
            #                     recharge = None

            #                     for i in [s]+self.NC:
            #                         if i != c:
            #                             for r in NR_:
            #                                 if x2[c,i,r,v,s].X > 0.9:
            #                                     next_node = i
            #                                     recharge = r
            #                                     break
            #                             if next_node is not None:
            #                                 break
                                
            #                     if recharge == '0':
            #                         route.append(next_node)
            #                     else:
            #                         route.append(recharge)
            #                         route.append(next_node)

            #                     c = next_node

            #                     if c == s:
            #                         break
            #                 print(v, route)                

            objf = model.ObjVal
            gap = model.MIPGap
            exe_time = model.Runtime

        return objf, gap, exe_time

    def graph_route(self, objf, gap):
        if objf == float("inf"):
            return False
        
        def extract_routes_1e():
            routes_1e = []
            for v in self.T:
                if self.y0[v] <= 0.9:
                    continue

                node = self.Nd
                route = [node]
                max_steps = len(self.NS) + 2

                for _ in range(max_steps):
                    next_node = None
                    for i in [self.Nd]+self.NS:
                        if i == node:
                            continue
                        if self.x0[node, i, v] > 0.9:
                            next_node = i
                            break

                    if next_node is None or next_node == self.Nd:
                        route.append(self.Nd)
                        break

                    route.append(next_node)
                    node = next_node

                if len(route) > 2:
                    routes_1e.append((v, route))
            return routes_1e
        
        def extract_routes_2e_and_labels():
            routes_2e = []
            arcs_r = {}

            for s in self.NS:
                if self.a[s] <= 0.9:
                    continue

                for v in self.F:
                    if self.y[v, s] <= 0.9:
                        continue

                    node = s
                    route = [s]
                    max_steps = len(self.NC) + 2

                    for _ in range(max_steps):
                        next_node = None
                        chosen_r = '0'

                        for i in [s] + self.NC:
                            if i == node:
                                continue

                            for r in ['0'] + self.NR:
                                if self.x[node, i, r, v, s] > 0.9:
                                    next_node = i
                                    chosen_r = r
                                    break

                            if next_node is not None:
                                break

                        if next_node is not None and chosen_r != '0':
                            arcs_r[(node, next_node)] = chosen_r

                        if next_node is None or next_node == s:
                            route.append(s)
                            break

                        route.append(next_node)
                        node = next_node

                    if len(route) > 2:
                        routes_2e.append((s, v, route))

            return routes_2e, arcs_r

        routes_1e = extract_routes_1e()
        routes_2e, arcs_r = extract_routes_2e_and_labels()

        G = nx.DiGraph()
        G.add_node(self.Nd, tipo="depot")
        for s in self.NS:
            G.add_node(s, tipo="satellite")
        for i in self.NC:
            G.add_node(i, tipo="customer")

        def build_positions():
            pos = {self.Nd: (0.0, 5.0)}
            n_routes2 = max(1, len(routes_2e))
            for r_idx, (s, v, route) in enumerate(routes_2e):
                customers = route[1:-1]
                n_c = len(customers)
                if n_c == 0:
                    continue

                cx = 5.0 * (r_idx - (n_routes2 - 1)/2.0)
                cy = 0.0
                R = 1.5

                for j, node in enumerate(customers):
                    angle = 2*math.pi*(j + 0.5) / n_c
                    pos[node] = (cx + R*math.cos(angle + math.pi/2),
                                 cy + R*math.sin(angle + math.pi/2))

            # Rango X (si no hay clientes, usa depot)
            xs = [p[0] for p in pos.values()]
            xmin = min(xs) - 2
            xmax = max(xs) + 2

            # Colocar satélites en una línea arriba de clientes
            nS = len(self.NS)
            for idx, s in enumerate(self.NS):
                x_s = xmin + (idx + 1)*((xmax - xmin)/(nS + 1))
                pos[s] = (x_s, 3.0)

            return pos, xmin, xmax

        pos, xmin, xmax = build_positions()

        plt.figure(figsize=(12, 6))

        plt.hlines(y=4.0, xmin=xmin, xmax=xmax, linestyles="dashed", colors="gray", linewidth=1.2)
        plt.hlines(y=2.0, xmin=xmin, xmax=xmax, linestyles="dashed", colors="gray", linewidth=1.2)

        plt.text(xmin + 1, 4.5, "Depósito", ha="center", va="bottom", fontsize=10)
        plt.text(xmin + 1, 3.5, "Satélites", ha="center", va="bottom", fontsize=10)
        plt.text(xmin + 1, 1.5, "Clientes", ha="center", va="bottom", fontsize=10)

        nx.draw_networkx_nodes(G, pos, nodelist=[self.Nd], node_shape="s",
                               node_color="darkturquoise", node_size=900,
                               edgecolors="black")
        nx.draw_networkx_nodes(G, pos, nodelist=self.NS, node_shape="D",
                               node_color="yellowgreen", node_size=900,
                               edgecolors="black")
        nx.draw_networkx_nodes(G, pos, nodelist=self.NC, node_shape="o",
                               node_color="lightblue", node_size=900,
                               edgecolors="black")
        nx.draw_networkx_labels(G, pos, font_size=10, font_weight="bold")

        for v, route in routes_1e:
            arcs = list(zip(route[:-1], route[1:]))
            G.add_edges_from(arcs)
            nx.draw_networkx_edges(
                G, pos, edgelist=arcs,
                arrowstyle="-|>", arrowsize=18, width=2.5,
                edge_color="tab:gray", connectionstyle="arc3,rad=0.12",
                style="dashed"
            )

        colors = plt.cm.Dark2(np.linspace(0, 1, max(1, len(routes_2e))))

        for color, (s, v, route) in zip(colors, routes_2e):
            arcs = list(zip(route[:-1], route[1:]))
            G.add_edges_from(arcs)
            nx.draw_networkx_edges(
                G, pos, edgelist=arcs,
                arrowstyle="-|>", arrowsize=18, width=2.5,
                edge_color=[color], connectionstyle="arc3,rad=0.00"
            )

        if arcs_r:
            nx.draw_networkx_edge_labels(
                G, pos, edge_labels=arcs_r, font_size=8, font_color="black",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8)
            )

        plt.text(0.97, 0.97, f"Costs = {objf:.0f}\nGap = {100*gap:.2f}%",
                 transform=plt.gca().transAxes, ha="right", va="bottom",
                 fontsize=14, bbox=dict(boxstyle="round,pad=0.3",
                                        facecolor="white", edgecolor="black",
                                        alpha=0.8))

        plt.title(f"Instancia: {self.instance_name}", fontsize=14)
        plt.axis("off")
        plt.show()

        return True

    def get_route_1st(self, order, ws, wv, q):
        Q = sum(q[s] for s in order)
        vt = [v for v in self.V1 if self.Q[v] >= Q]

        if not vt:
            return None, None, float("inf"), 0

        route = [self.Nd, *order, self.Nd]

        best_route, best_v = None, None
        best_costs, best_duals = float("inf"), 0

        for v in vt:
            c = self.c[v]
            costs, duals = self.h[v], wv[v]
            duals += sum(ws[s] for s in order)

            for i, j in zip(route, route[1:]):
                costs += c*self.d[i,j]

            if (costs - duals) < (best_costs - best_duals):
                best_route, best_v = route, v
                best_costs, best_duals = costs, duals

        if best_route is None:
            return None, None, float("inf"), 0
        else:
            return best_route, best_v, best_costs, best_duals
        
    def LNS_1st(self, Q, a, ws, wv):
        NS = [s for s in self.NS if a[s] > 0.9]

        if not NS:
            return None, None, 0

        S_in, S_out = [], []
        for i in NS:
            (S_in if random.random() < 0.5 else S_out).append(i)

        random.shuffle(S_in)
        best_solution = self.get_route_1st(S_in, ws, wv, Q)

        if best_solution[0] is None:
            while S_in:
                s = random.choice(S_in)
                S_in.remove(s)
                S_out.append(s)
                best_solution = self.get_route_1st(S_in, ws, wv, Q)
                if best_solution[0] is not None:
                    break
        
        if best_solution[0] is None:
            return None, None, None

        p = self.pq
        max_ite = self.max
        not_improve = 0

        while not_improve < max_ite:
            # Remove
            if S_in:
                n = min(round(p*len(S_in)), len(S_in)-1)
                S_rem = random.sample(S_in, n)

                for s in S_rem: 
                    S_in.remove(s) 
                    S_out.append(s)
                
            # Insert
            random.shuffle(S_out)

            curr_solution = self.get_route_1st(S_in, ws, wv, Q)
            curr_S = S_in.copy()

            for s_ in S_out:
                for s in range(len(S_in) + 1):
                    S_new = S_in[:s] + [s_] + S_in[s:]
                    new_solution = self.get_route_1st(S_new, ws, wv, Q)

                    if new_solution[2] - new_solution[3] < curr_solution[2] - curr_solution[3]:
                        curr_solution, curr_S = new_solution, S_new.copy()

                S_in = curr_S.copy()

            S_out = [s for s in S_out if s not in S_in]
   
            if curr_solution[2] - curr_solution[3] < best_solution[2] - best_solution[3]:
                best_solution = curr_solution
                not_improve = 0
            else:
                not_improve += 1

        if best_solution[2] - best_solution[3] < 0:
            return best_solution[0], best_solution[1], best_solution[2]
        else:
            return None, None, None
    
    def set_partitioning_1st(self, w, env_params):
        with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            env = gp.Env(params=env_params)

        # Model
        sp_model = gp.Model("Subproblem: Set partitioning (1st level)", env=env)
        sp_model.Params.OutputFlag = 0

        # Demand per stop
        Q = {s: sum(self.q[i]*w[i,s] for i in self.NC) for s in self.NS}

        # Variables
        R_ = []
        x = {}

        for r in self.routes[self.Nd]:
            Rd_ = self.Rd[r,self.Nd]
            q = sum(Q[s] for s in Rd_[1:-1])

            cap = sum(self.Q[v]*self.b[r,v,self.Nd] for v in self.V1)

            if q <= cap:
                R_.append(r)
                x[r] = sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, ub = 1, name = f"x({r})")

        y = {v: sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"y({v})") for v in self.V1}

        #Objective function
        M = 10**(math.ceil(math.log10(len(self.NS)*max(self.C.values())))+1)

        sp_model.setObjective(gp.quicksum(self.C[r,self.Nd]*x[r] for r in R_) + gp.quicksum(M*y[v] for v in self.V1), gp.GRB.MINIMIZE)

        #Constraints
        LocCtr_c = {}
        LocCtr_v = {}

        a = {s: max(w[i,s] for i in self.NC) for s in self.NS}

        for s in self.NS:
            LocCtr_c[s] = sp_model.addConstr(gp.quicksum(self.A[s,r,self.Nd]*x[r] for r in R_) == a[s])

        for v in self.V1:
            LocCtr_v[v] = sp_model.addConstr(gp.quicksum(self.b[r,v,self.Nd]*x[r] for r in R_) <= self.m[v] + y[v])

        Optimal = False

        while not Optimal:
            sp_model.update()
            sp_model.optimize()

            Optimal = True
            
            ws = {s: LocCtr_c[s].Pi for s in self.NS}
            wv = {v: LocCtr_v[v].Pi for v in self.V1}

            new_r, veh, cost = self.LNS_1st(Q, a, ws, wv)

            if new_r is not None:
                r_id = f"r{len(self.routes[self.Nd])}"
                self.routes[self.Nd].append(r_id)
                R_.append(r_id)
                self.Rd[r_id,s] = new_r
                self.C[r_id,s] = cost
                Optimal = False

                newCol = gp.Column()

                for s in self.NS:
                    self.A[s,r_id,self.Nd] = 1 if s in new_r else 0
                    newCol.addTerms(self.A[s,r_id,self.Nd], LocCtr_c[s])

                for v in self.V1:
                    self.b[r_id,v,self.Nd] = 0 if v != veh else 1
                    newCol.addTerms(self.b[r_id,v,self.Nd], LocCtr_v[v])

                x[r_id] = sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, ub = 1, obj=cost, name = f"x({r_id})", column = newCol)

        for r in R_:
            x[r].setAttr("vtype", gp.GRB.BINARY)
                
        sp_model.update()
        sp_model.optimize()

        if sum(y[v].X for v in self.V1) > 0:
            return None, float("inf")
        
        r_ = []
        for r in R_:
            if x[r].X > 0.9:
                r_.append(r)
                
        return r_, sp_model.ObjVal

    def elecPULSE(self, order, v):
        class PULSE:
            def __init__(self, order, NR, d, t, e, l, S, p, c, L, r, eta):
                self.order = order
                self.stations = NR
                self.d = d
                self.t = t
                self.e = e
                self.l = l
                self.S = S
                self.p = p
                self.c = c
                self.L = L
                self.r = r
                self.eta = eta

                self.lb = {n: 0.0 for n in range(len(order))}
                for n in range(len(self.order) - 2, -1, -1):
                    i, j = order[n], order[n + 1]
                    self.lb[n] = self.lb[n + 1] + self.c*self.d[i,j]

                self.cost_battery = {n: [(float("inf"), 0)] for n in range(len(order))}

                self.ub = float("inf")
                self.best_order = None

            def pulse(self, n, cost, time, battery, path):
                if battery < 0:
                    return
                else:
                    if cost + self.lb[n] >= self.ub:
                        return
                    else:
                        for c_, b_ in self.cost_battery[n]:
                            if (c_ < cost and b_ >= battery) or (c_ <= cost and b_ > battery):
                                return

                        self.cost_battery[n] = [(c, b) for c, b in self.cost_battery[n] if not ((cost < c and battery >= b) or (cost <= c and battery > b))]
                        self.cost_battery[n].append((cost, battery))

                        if n == len(order) - 1:
                            if cost < self.ub:
                                self.ub = cost
                                self.best_order = path.copy()
                            return

                        i, j = order[n], order[n + 1]

                        # Direct
                        if n + 1 == len(order) - 1:
                            new_time = float("inf")
                            new_cost = cost + self.c*self.d[i,j]
                            new_battery = battery - self.r*self.d[i,j]

                            self.pulse(n + 1, new_cost, new_time, new_battery, path + [j])
                        else:
                            new_time = max(time + self.t[i,j], self.e[j])
                            new_cost = cost + self.c*self.d[i,j] + self.p[j]*max(new_time - self.l[j], 0)
                            new_time += self.S[j]
                            new_battery = battery - self.r*self.d[i,j]

                            self.pulse(n + 1, new_cost, new_time, new_battery, path + [j])

                        # Recharge stations
                        for r in self.stations:
                            if battery >= self.r*self.d[i,r]:
                                if n + 1 == len(order) - 1:
                                    new_time = float("inf")
                                    new_cost = cost + self.c*(self.d[i,r] + self.d[r,j])
                                    new_battery = self.L - self.eta*self.d[r,j]
                                
                                    self.pulse(n + 1, new_cost, new_time, new_battery, path + [r,j])
                                else:
                                    new_time = max(time + self.t[i,r] + self.eta*(self.L - (battery - self.r*self.d[i,r])) + self.t[r,j], self.e[j])
                                    new_cost = cost + self.c*(self.d[i,r] + self.d[r,j]) + self.p[j]*max(new_time - self.l[j], 0)
                                    new_time += self.S[j]
                                    new_battery = self.L - self.eta*self.d[r,j]
                                
                                    self.pulse(n + 1, new_cost, new_time, new_battery, path + [r,j])

            def solve(self):
                self.pulse(0, 0, 0, self.L, [self.order[0]])
                return self.ub, self.best_order

        ePULSE = PULSE(order, self.NR, self.d, self.t, self.e, self.l, self.S, self.p, self.c[v], self.L[v], self.r[v], self.eta)
        cost, orderF = ePULSE.solve()
        cost += self.h[v]

        return cost, orderF

    def get_route_2nd(self, s, order, wc, wv):
        Q = sum(self.q[i] for i in order)    
        vt = [v for v in self.V2 if self.Q[v] >= Q]

        if not vt:
            return None, None, float("inf"), 0

        base = [s,*order,s]

        best_route, best_v = None, None
        best_costs, best_duals = float("inf"), 0

        for v in vt:
            costs, route = self.elecPULSE(base, v)
            duals = wv[v]

            for i in order:
                duals += wc[i]

            if (costs - duals) < (best_costs - best_duals):
                best_route, best_v = route, v
                best_costs, best_duals = costs, duals

        if best_route is None:
            return None, None, float("inf"), 0

        return best_route, best_v, best_costs, best_duals
            
    def ALNS_2nd(self, s, w, wc, wv):
        NC = [i for i in self.NC if w[i,s] > 0.9]
        if not NC:
            return None, None, 0

        C_in, C_out = [], []
        for i in NC:
            (C_in if random.random() < 0.5 else C_out).append(i)

        random.shuffle(C_in)

        best_solution = self.get_route_2nd(s, C_in, wc, wv)

        if best_solution[0] is None:
            while C_in:
                c = random.choice(C_in)
                C_in.remove(c)
                C_out.append(c)
                best_solution = self.get_route_2nd(s, C_in, wc, wv)
                if best_solution[0] is not None:
                    break
        
        if best_solution[0] is None:
            return None, None, None

        p = self.pq
        max_ite = self.max
        not_improve = 0

        while not_improve < max_ite:
            Hmax = max(self.H[o] for o in self.op)
            pi = {o: math.exp(self.H[o]-Hmax)/sum(math.exp(self.H[o_]-Hmax) for o_ in self.op) for o in self.op}
            rem, ins = random.choices(list(pi.keys()), weights=list(pi.values()))[0]

            # Remove
            if C_in:
                n = min(round(p*len(C_in)), len(C_in)-1)

                if rem == 'Rnd':
                    C_rem = random.sample(C_in, n)
                elif rem == 'Dist-r':
                    seed = random.choice(C_in)
                    C_rem = sorted(C_in, key=lambda i: self.d[seed,i])[:n]
                elif rem == 'W-Dist':
                    dist = {c: self.d[s,c] if i == 0 else self.d[C_in[i-1],c] for i, c in enumerate(C_in)}
                    C_rem = sorted(C_in, key=lambda i: dist[i], reverse=True)[:n]
                elif rem == 'TW-r':
                    seed = random.choice(C_in)
                    C_rem = sorted(C_in, key=lambda i: abs(self.l[seed]-self.l[i]))[:n]
                elif rem == 'W-Lat':
                    lat, t = {}, 0
                    for i in range(len(C_in)):
                        t = max(self.e[C_in[i]], t + (self.t[s,C_in[i]] if i == 0 else self.S[C_in[i-1]] + self.t[C_in[i-1],C_in[i]]))
                        lat[C_in[i]] = max(t - self.l[C_in[i]], 0)
                    C_rem = sorted(C_in, key=lambda i: lat[i], reverse=True)[:n]

                for c in C_rem: 
                    C_in.remove(c) 
                    C_out.append(c)
                
            # Insert
            if ins == 'Rnd':
                random.shuffle(C_out)
            elif ins == 'Dist':
                C_out = sorted(C_out, key=lambda i: self.d[s,i])
            elif ins == 's-TW':
                C_out = sorted(C_out, key=lambda i: self.e[i])
            elif ins == 'e-TW':
                C_out = sorted(C_out, key=lambda i: self.l[i])

            curr_solution = self.get_route_2nd(s, C_in, wc, wv)
            curr_C = C_in.copy()

            for c_ in C_out:
                for i in range(len(C_in) + 1):
                    C_new = C_in[:i] + [c_] + C_in[i:]
                    new_solution = self.get_route_2nd(s, C_new, wc, wv)

                    if new_solution[2] - new_solution[3] < curr_solution[2] - curr_solution[3]:
                        curr_solution, curr_C = new_solution, C_new.copy()

                C_in = curr_C.copy()

            C_out = [c for c in C_out if c not in C_in]

            # Obtain best solution
            if curr_solution[2] - curr_solution[3] < best_solution[2] - best_solution[3]:
                dH = 1 - self.R
                best_solution = curr_solution
                not_improve = 0
            else:
                dH = -1 - self.R
                not_improve += 1

            # Update MAB
            for o in self.op:
                if o == (rem, ins):
                    self.H[o] += self.alpha*dH*(1-pi[o])
                else:
                    self.H[o] -= self.alpha*dH*pi[o]
                        
            self.n += 1
            self.R += dH/self.n

        if best_solution[2] - best_solution[3] < 0:
            return best_solution[0], best_solution[1], best_solution[2]
        else:
            return None, None, None

    def set_partitioning_2nd(self, w, env_params):
        with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            env = gp.Env(params=env_params)

        #Model
        sp_model = gp.Model("Subproblem: Set partitioning (2nd level)", env=env)
        sp_model.Params.OutputFlag = 0

        #Variables
        x = {(r,s): sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, ub = 1, name = f"x^{s}({r})") for s in self.NS for r in self.routes[s]}
        a = {(i,s): sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, ub = 1, name = f"a({i})") for i in self.NC for s in self.NS}
        ys = {(v,s): sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"y^{s}({v})") for s in self.NS for v in self.V2}
        y = {v: sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"y({v})") for v in self.V2}

        #Objective function
        M = 10**(math.ceil(math.log10(len(self.NC)*max(self.C.values())))+1)
        sp_model.setObjective(gp.quicksum(self.C[r,s]*x[r,s] for s in self.NS for r in self.routes[s]) + gp.quicksum(M*a[i,s] for i in self.NC for s in self.NS)
                              + gp.quicksum(M*ys[v,s] for s in self.NS for v in self.V2) + gp.quicksum(M*y[v] for v in self.V2), gp.GRB.MINIMIZE)

        #Constraints
        LocCtr_c = {}
        LocCtr_vs = {}
        LocCtr_v = {}

        for s in self.NS:
            for i in self.NC:
                LocCtr_c[i,s] = sp_model.addConstr(gp.quicksum(self.A[i,r,s]*x[r,s] for r in self.routes[s]) == w[i,s] - a[i,s])

            for v in self.V2:
                LocCtr_vs[v,s] = sp_model.addConstr(gp.quicksum(self.b[r,v,s]*x[r,s] for r in self.routes[s]) <= self.ms[v] + ys[v,s])

        for v in self.V2:
            LocCtr_v[v] = sp_model.addConstr(gp.quicksum(self.b[r,v,s]*x[r,s] for s in self.NS for r in self.routes[s]) <= self.m[v] + y[v])

        Optimal = False

        while not Optimal:
            sp_model.update()
            sp_model.optimize()

            Optimal = True
            
            for s in self.NS:
                wc = {i: LocCtr_c[i,s].Pi for i in self.NC}
                wv = {v: LocCtr_vs[v,s].Pi + LocCtr_v[v].Pi for v in self.V2}

                new_r, veh, cost = self.ALNS_2nd(s, w, wc, wv)

                if new_r is not None:
                    r_id = f"r{len(self.routes[s])}"
                    self.routes[s].append(r_id)
                    self.Rd[r_id,s] = new_r
                    self.C[r_id,s] = cost
                    Optimal = False

                    newCol = gp.Column()

                    for i in self.NC:
                        self.A[i,r_id,s] = 1 if i in new_r else 0
                        newCol.addTerms(self.A[i,r_id,s], LocCtr_c[i,s])

                    for v in self.V2:
                        self.b[r_id,v,s] = 0 if v != veh else 1
                        newCol.addTerms(self.b[r_id,v,s], LocCtr_vs[v,s])
                        newCol.addTerms(self.b[r_id,v,s], LocCtr_v[v])

                    x[r_id,s] = sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, ub = 1, obj=cost, name = f"x^{s}({r_id})", column = newCol)

        for s in self.NS:
            for r in self.routes[s]:
                x[r,s].setAttr("vtype", gp.GRB.BINARY)
                
        sp_model.update()
        sp_model.optimize()

        if sum(y[v].X for v in self.V2) > 0 or sum(ys[v,s].X for s in self.NS for v in self.V2) > 0 or sum(a[i,s].X for i in self.NC for s in self.NS):
            return {s: None for s in self.NS}, float("inf")

        r_ = {}
        for s in self.NS:
            r_s = []
            for r in self.routes[s]:
                if x[r,s].X > 0.9:
                    r_s.append(r)
            r_[s] = r_s
                
        return r_, sp_model.ObjVal

    def init_math(self, env_params, maxQ):
        with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            env = gp.Env(params=env_params)

        #Model
        w_model = gp.Model("Initialize", env=env)
        w_model.Params.OutputFlag = 0

        #Variables
        w = {(i,s): w_model.addVar(vtype = gp.GRB.BINARY, lb = 0, ub = 1, name = f"w^{i}({s})") for i in self.NC for s in self.NS}

        #Objective function
        w_model.setObjective(gp.quicksum(self.d[s,i]*w[i,s] for i in self.NC for s in self.NS), gp.GRB.MINIMIZE)

        #Constraints
        for i in self.NC:
            w_model.addConstr(gp.quicksum(w[i,s] for s in self.NS) == 1) #Satellite - Customer

        for s in self.NS:
            w_model.addConstr(gp.quicksum(self.q[i]*w[i,s] for i in self.NC) <= maxQ) #Capacity

        w_model.update()
        w_model.optimize()
            
        w_ = {(i,s): round(w[i,s].X) for s in self.NS for i in self.NC}
                
        return w_

    def matheuristic(self, params, exe_time):
        start = time.perf_counter()

        # Initialize routes
        # 1st level routes
        r = []
        for s in self.NS:
            for v in self.V1:
                rdx = len(r) if r else 0
                r_id = f"r{rdx}"
                    
                r.append(r_id)
                self.Rd[r_id,self.Nd] = [self.Nd,s,self.Nd]
                self.C[r_id,self.Nd] = self.h[v] + self.c[v]*(self.d[self.Nd,s] + self.d[s,self.Nd])

                for v_ in self.V1:
                    self.b[r_id,v_,self.Nd] = 0 if v != v_ else 1

                for i in self.NS:
                    self.A[i,r_id,self.Nd] = 0 if i != s else 1
        self.routes[self.Nd] = r

        # 2nd level routes
        for s in self.NS:
            r = []
            for i in self.NC:
                for v in self.V2:
                    if self.Q[v] >= self.q[i]:
                        cost, order = self.elecPULSE([s,i,s],v)
                        
                        if cost != float("inf"):
                            rdx = len(r) if r else 0
                            r_id = f"r{rdx}"

                            r.append(r_id)
                            self.Rd[r_id,s] = order
                            self.C[r_id,s] = cost

                            for v_ in self.V2:
                                self.b[r_id,v_,s] = 0 if v != v_ else 1
                    
                            for j in self.NC:
                                self.A[j,r_id,s] = 0 if i != j else 1

            self.routes[s] = r

        # Initialize w's
        maxQ = max(self.Q.values())
        w = self.init_math(params, maxQ)

        # Set partitioning
        rs, z = self.set_partitioning_2nd(w, params)
        rD, zD = self.set_partitioning_1st(w, params)

        rs[self.Nd] = rD
        z += zD

        best_rs = rs.copy()
        best_z = z

        improved = True

        while improved:
            improved = False

            if time.perf_counter() - start > exe_time:
                return best_z, best_rs, time.perf_counter() - start
            
            sat = {i: None for i in self.NC}
            q = {s: 0 for s in self.NS}

            for s in self.NS:
                for i in self.NC:
                    if w[i,s] > 0.9:
                        sat[i] = s
                        q[s] += self.q[i]

            w_change = None
            rs_change = None
            z_change = float("inf")

            for i in self.NC:
                curr_s = sat[i]

                for s in self.NS:
                    if time.perf_counter() - start < exe_time:
                        if s != curr_s and q[s] + self.q[i] <= maxQ:
                            w[i,s] = 1
                            w[i,curr_s] = 0
                            z = 0

                            if time.perf_counter() - start < exe_time:
                                rs, z = self.set_partitioning_2nd(w, params) 
                            else:
                                z = float("inf")

                            if time.perf_counter() - start < exe_time:
                                rD, zD = self.set_partitioning_1st(w, params)  
                                rs[self.Nd] = rD
                                z += zD
                            else:
                                z = float("inf")
                                    
                            if z < z_change:
                                z_change = z
                                w_change = w.copy()
                                rs_change = rs.copy()

                            w[i,s] = 0
                            w[i,curr_s] = 1

            for i, j in itertools.combinations(self.NC, 2):
                if time.perf_counter() - start < exe_time:
                    si = sat[i]
                    sj = sat[j]

                    if si != sj and q[si] - self.q[i] + self.q[j] <= maxQ and q[sj] - self.q[j] + self.q[i] <= maxQ:
                        w[i,si], w[i,sj] = 0, 1
                        w[j,si], w[j,sj] = 1, 0
                        z = 0

                        if time.perf_counter() - start < exe_time:
                            rs, z = self.set_partitioning_2nd(w, params)
                        else:
                            z = float("inf")

                        if time.perf_counter() - start < exe_time:
                            rD, zD = self.set_partitioning_1st(w, params)  
                            rs[self.Nd] = rD
                            z += zD
                        else:
                            z = float("inf")

                        if z < z_change:
                            z_change = z
                            w_change = w.copy()
                            rs_change = rs.copy()

                        w[i,si], w[i,sj] = 1, 0
                        w[j,si], w[j,sj] = 0, 1

            if z_change < best_z:
                best_z = z_change
                w = w_change.copy()
                best_rs = rs_change.copy()
                improved = True

        end = time.perf_counter()

        return best_z, best_rs, end - start

In [ ]:
params = {
    "WLSACCESSID": '...',
    "WLSSECRET": '...',
    "LICENSEID": 0000
}

folder = "H1/Set2"
file = "H1-E-Set2a_E-n22-k4-s6-17_int"

e2evrp = E2EVRP(folder, file)
of_milp, g, t_milp = e2evrp.milp_model(params, 1800, 0)
print("MILP:", "Obj.Function:", of_milp, "Exe. time:", t_milp, "\n")
of_math, r, t_math = e2evrp.matheuristic(params, 1800)
print("Matheuristic:", "Obj.Function:", of_math, "Exe. time:", t_math)

First level:
vT0 ['D0', 'S0', 'D0']
Second level:
S0
vF1 ['S0', 'R0', 'C4', 'C3', 'C5', 'C2', 'R0', 'S0']
vF2 ['S0', 'R0', 'C0', 'R0', 'C1', 'C8', 'C9', 'C6', 'C7', 'S0']
MILP: Obj.Function: 3023.0 Exe. time: 0.7419741153717041 

Matheuristic: Obj.Function: 3023.0 Exe. time: 2.711976999999024
